# Sentiment Analysis Pipeline

This notebook explores sentiment analysis with two approaches:

- VADER for rule-based social-media sentiment classification
- Naive Bayes with bag-of-words and TF-IDF features for supervised sentiment classification

The workflow includes preprocessing experiments, classification reports, error analysis, and feature inspection.

###   Collecting 50 tweets for evaluation
Collect 50 tweets. Try to find tweets that are interesting for sentiment analysis, e.g., very positive, neutral, and negative tweets. These could be your own tweets (typed in) or collected from the Twitter stream.

We will store the tweets in the file **my_tweets.json** (use a text editor to edit).
For each tweet, you should insert:
* sentiment analysis label: negative | neutral | positive (this you determine yourself, this is not done by a computer)
* the text of the tweet
* the Tweet-URL

from:
```
    "1": {
        "sentiment_label": "",
        "text_of_tweet": "",
        "tweet_url": "",
```
to:
```
"1": {
        "sentiment_label": "positive",
        "text_of_tweet": "All across America people chose to get involved, get engaged and stand up. Each of us can make a difference, and all of us ought to try. So go keep changing the world in 2018.",
        "tweet_url" : "https://twitter.com/BarackObama/status/946775615893655552",
    },
```

You can load your tweets with human annotation in the following way.

In [1]:
import json

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
tweets_path = PROJECT_ROOT / "data" / "raw" / "my_tweets.json"
my_tweets = json.load(open(tweets_path, encoding="utf-8"))

In [3]:
for id_, tweet_info in my_tweets.items():
    print(id_, tweet_info)
    break

1 {'sentiment_label': 'Negative', 'text_of_tweet': 'Can’t trust a person who likes dark chocolate', 'tweet_url': 'https://twitter.com/Gabexhb/status/1653806986251624448'}


### Findings
#### 3a quantitative analysis
In total the model predicted 32/50 tweet tones with the same outcome as us. 
The model had the following number of errors for each type of classification we made:
- 3/18 for our positive classifications
- 10/18 for our neutral classifications
- 5/18 for our negative classifications

As we can see the vader model had a the most trouble classifying the neutral posts correctly.

For each prediction we made we list the number of errors made and which classification was used.
Our Classification |  model

Positive           |  ____________  | Neutral  1/3 |  Negative 2/3

Neutral            |   Positive 6/10 | ____________ |   Negative 4/10

Negative           |   Positive 2/6  | Neutral  4/6 | ____________

We can see that  there was a even distribution of incorrect classifications for each type.
 
#### 3b Error Analysis
#### Incorrect guesses when we had Positive:
Total: 3
- Neu: 1
- Neg: 2

---- 

Rarely seen timely captured by photographer
neutral

"rarely seen" caused a problem with vader as it can be seen as slightly negative which can result in neutral classification as timely captured will be slightly positive. 

---- 

No matter how many new desserts come out nothing beats the OG lotus softie from salt
negative

"nothing beats" for us can be seen as a strong positive however the vader classification could take the "nothing" as a negative indication

---- 

Ive called many 20X, 30X, 50X and even 1000X plays. Now its time to educate you guys on how to do it yourself. Gonna drop threads on finding the next 50-100x coins.Ready? 
negative

The classifier associate the numbers to certain risk and therefore return a negative classification.

### Incorrect guesses when we had Neutral:
Total: 10
- Pos : 6
- Neg : 4

I was gonna tweet something else but I like this one more
Positive

You can see that the use of words such as like and more would make vader lean towards positive. we labeled it as neutral due to the fact that is is not a directly positive sentence

---

What were you doing if you weren’t playing Lego video games
Positive

Vader takes the enthusiasm for in this tweet as a positive when we see it as a neutral question

---

You got to get the outfit ready before the first day of the school year
positive

imperitive tone can be seen as a positive while we see it as a neutral statement

----

The issues with the platform gap, any ideas for a solution?
positive

Vader could see this as a positive as you are looking for a request of ideas and there is collaborations however we saw it as neutral as it is a simple ask for solutions

----

Artificial intelligence was asked to make a picture of Mother Teresa fighting against poverty.
negative

AI ha a negative sentiment as well as the discussion of poverty.

----
Jean Arnault, son of one of the richest men in the world, checking out vintage cars
positive

Wealth and vintage cars could be seen as a positive thing while this is a neutral statement.

----
Bill Gates jumps over a chair in his 1994 interview with Connie Chung. Gates was the richest person in the world at the time of the interview with a net worth of over $9B.
positive

Bill gates and his success ccould be seen as a positive thing as well as jumping over your chair "with joy" while this is a neutral statement

----
What happened to that new top boy season supposed to drop
negative

Can be seen as negaitve due to the anticipation / the new season has not been 'dropped' yet.

----
Nothing is scarier than a reply with a gif from the show “friends”
neutral

scary is seen as negative while it is quite a neutral statement.

----
What always drives me crazy is that Biden does normal shit like eat ice cream and ride a bike while Trump practically lived in ivory tower with a solid gold toilet built on the back of hundreds of millions of dollars of fraud and somehow Biden isn't viewed as a man of the people.
negative

Use of fraud etc. in addition to  the comparison of the two suggesting one is better than another. While we see it as a neutral comparison of the two. 

### Incorrect guessess when we had Negative:
Total:6
- Pos: 2
- Neu: 4

Can’t trust a person who likes dark chocolate
Positive
Sceptisism could be seen as a positive thing as well as the use of words like 'likes'

---

I need a girl to break my heart so I can start going to the gym
neutral
Neutral as going to the gym can balance out the broken heart. We see it as self degredation.

----

This is one of most hilarious arts I have ever seen
positive

Humor can be seen as a positive while we saw it as a negative due to them making fun of the arwork in further comments

----

Imagine the valet guy crashes your car
neutral
Can be seen as neutral due to the vader thinking it is a imaginaty scenario

----
Static fire test of our next @NASA astronaut flight
negative

Static fire can be seen as negative when in context with a flight.

### Analysis of Results

As we can see, the model made the most mistakes on the tweets that we labeled as neutral. this could be because vader has trouble not getting a slight inclenation towards either negative or positive.

In [4]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
nlp = spacy.load('en_core_web_sm')
vader_model = SentimentIntensityAnalyzer()

def run_vader(textual_unit,
              lemmatize=False,
              parts_of_speech_to_consider=None,
              verbose=0):
    """
    Run VADER on a sentence from spacy

    :param str textual unit: a textual unit, e.g., sentence, sentences (one string)
    (by looping over doc.sents)
    :param bool lemmatize: If True, provide lemmas to VADER instead of words
    :param set parts_of_speech_to_consider:
    -None or empty set: all parts of speech are provided
    -non-empty set: only these parts of speech are considered.
    :param int verbose: if set to 1, information is printed
    about input and output

    :rtype: dict
    :return: vader output dict
    """
    doc = nlp(textual_unit)

    input_to_vader = []

    for sent in doc.sents:
        for token in sent:

            to_add = token.text

            if lemmatize:
                to_add = token.lemma_

                if to_add == '-PRON-':
                    to_add = token.text

            if parts_of_speech_to_consider:
                if token.pos_ in parts_of_speech_to_consider:
                    input_to_vader.append(to_add)
            else:
                input_to_vader.append(to_add)

    scores = vader_model.polarity_scores(' '.join(input_to_vader))

    if verbose >= 1:
        print()
        print('INPUT SENTENCE', sent)
        print('INPUT TO VADER', input_to_vader)
        print('VADER OUTPUT', scores)

    return scores

In [5]:
def vader_output_to_label(vader_output):
    """
    map vader output e.g.,
    {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.4215}
    to one of the following values:
    a) positive float -> 'positive'
    b) 0.0 -> 'neutral'
    c) negative float -> 'negative'

    :param dict vader_output: output dict from vader

    :rtype: str
    :return: 'negative' | 'neutral' | 'positive'
    """
    compound = vader_output['compound']

    if compound < 0:
        return 'negative'
    elif compound == 0.0:
        return 'neutral'
    elif compound > 0.0:
        return 'positive'

assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.0}) == 'neutral'
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.01}) == 'positive'
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': -0.01}) == 'negative'

In [6]:
tweets = []
all_vader_output = []
gold = []

# settings (to change for different experiments)
to_lemmatize = True
pos = set()

for id_, tweet_info in my_tweets.items():
    the_tweet = tweet_info['text_of_tweet']
    vader_output = run_vader(the_tweet) # run vader
    vader_label = vader_output_to_label(vader_output) # convert vader output to category

    tweets.append(the_tweet)
    all_vader_output.append(vader_label)
    gold.append(tweet_info['sentiment_label'])

# use scikit-learn's classification report
for i, tweet in enumerate(tweets):
    print()
    print(tweet)
    print("Vader:", all_vader_output[i])
    print("Annotation:", gold[i])
    


Can’t trust a person who likes dark chocolate
Vader: positive
Annotation: Negative

Im a Dallas fan now
Vader: positive
Annotation: Positive

21 savage is the greatest lyricist of all time with this bar
Vader: positive
Annotation: Positive

I was gonna tweet something else but I like this one more
Vader: positive
Annotation: Neutral

Sometimes I do forget that we are some of the most privileged people on the planet. Private school, never had to work while studying, going uni abroad. Never acc worried about our next meal. Warm house. Safe neighborhoods.
Vader: positive
Annotation: Positive

What were you doing if you weren’t playing Lego video games
Vader: positive
Annotation: Neutral

I need a girl to break my heart so I can start going to the gym
Vader: neutral
Annotation: Negative

Pitbull is in the hall of fame of music.
Vader: positive
Annotation: Positve

Even copying off someone’s homework is too much effort
Vader: neutral
Annotation: Neutral

You got to get the outfit ready bef

#### Findings
The classification reports indicates for us the precision, recall, f1 scores per category as well as the macro and micro averages. Each experiment for the VADER analysis are set under different conditions, whether they are lemmatized or not, only verbs, nouns, etc. This tells us the impact it has on VADER when analyzing the airline tweets.

- Does lemmatisation help?
In general lemmatisation helps to reduce words to a standardized form which can help reduce the dimensions of the feature space. However when analyzing the results below, it can be observed that there was not a significant impact on the results when using lemmatization or not. Only a slight increase/decrease on the specific score for negative, neutral, positive conditions. The reasoning for this could be because VADER already analyzes the variations of words well enough and doesnt need lemmatization to simplify the words, this might only lead to losing context of the information when conduction the sentiment analysis.

- Are all parts of speech equally important for sentiment analysis?
All parts of the speech contribute to the score for sentiment analysis equally. This is because without all context for the sentence/speech, important information may get discarded causing a significant change in the overall results. Any adjective, noun or verb could impact the tone/mood for any part of speech. It can be observed that recall for the VADER predicted labels falls dramatically when only using one part of speech instead of all. And similarly f1 scores also have a decrease across all classes. It should be noted, however, that this decrease is also different depending on the different parts of speech used, but it can range from 0.05 up to 0.3 in score. This results show that to get the most accurate classification prediction using VADER all parts of speech are important. Further experiments could be done combining multiple parts of speech such as adjectives and nouns, instead of using only a single part of speech per experiment, to further analyse the importance of the different parts of speech.

In [7]:
# Load airline tweet files from the project data directory

import os
import pathlib
from sklearn.datasets import load_files
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
airline_tweets_folder = PROJECT_ROOT / "data" / "raw" / "airlinetweets"
airline_tweets_train = load_files(str(airline_tweets_folder))

In [8]:
import numpy as np
def classification_report_own(negative_tweets_annotations, positive_tweets_annotations, neutral_tweets_annotations):
    true_negative = [1 for annot in negative_tweets_annotations]
    true_positive = [1 for annot in positive_tweets_annotations]
    true_neutral = [1 for annot in neutral_tweets_annotations]

    predicted_neg = [1 if annot == 'negative' else 0 for annot in negative_tweets_annotations]
    predicted_pos = [1 if annot == 'positive' else 0 for annot in positive_tweets_annotations]
    predicted_neutral = [1 if annot == 'neutral' else 0 for annot in neutral_tweets_annotations]

    true_positive_count_neg = sum([1 for true_label, predicted_label in zip(true_negative, predicted_neg) if true_label == 1 and predicted_label == 1])
    true_negative_count_neg = sum([1 for true_label, predicted_label in zip(true_negative, predicted_neg) if true_label != 1 and predicted_label != 1])
    false_positive_count_neg = sum([1 for true_label, predicted_label in zip(true_negative, predicted_neg) if true_label != 1 and predicted_label == 1])
    false_negative_count_neg = sum([1 for true_label, predicted_label in zip(true_negative, predicted_neg) if true_label == 1 and predicted_label != 1])

    true_positive_count_pos = sum([1 for true_label, predicted_label in zip(true_positive, predicted_pos) if true_label == 1 and predicted_label == 1])
    true_negative_count_pos = sum([1 for true_label, predicted_label in zip(true_positive, predicted_pos) if true_label != 1 and predicted_label != 1])
    false_positive_count_pos = sum([1 for true_label, predicted_label in zip(true_positive, predicted_pos) if true_label != 1 and predicted_label == 1])
    false_negative_count_pos = sum([1 for true_label, predicted_label in zip(true_positive, predicted_pos) if true_label == 1 and predicted_label != 1])

    true_positive_count_neutral = sum([1 for true_label, predicted_label in zip(true_neutral, predicted_neutral) if true_label == 1 and predicted_label == 1])
    true_negative_count_neutral = sum([1 for true_label, predicted_label in zip(true_neutral, predicted_neutral) if true_label != 1 and predicted_label != 1])
    false_positive_count_neutral = sum([1 for true_label, predicted_label in zip(true_neutral, predicted_neutral) if true_label != 1 and predicted_label == 1])
    false_negative_count_neutral = sum([1 for true_label, predicted_label in zip(true_neutral, predicted_neutral) if true_label == 1 and predicted_label != 1])

    # Precision
    neg_precision = precision_score(true_negative, predicted_neg)
    pos_precision = precision_score(true_positive, predicted_pos)
    neutral_precision = precision_score(true_neutral, predicted_neutral)

    # Recall
    neg_recall = recall_score(true_negative, predicted_neg)
    pos_recall = recall_score(true_positive, predicted_pos)
    neutral_recall = recall_score(true_neutral, predicted_neutral)

    # F1 score
    neg_f1 = f1_score(true_negative, predicted_neg)
    pos_f1 = f1_score(true_positive, predicted_pos)
    neutral_f1 = f1_score(true_neutral, predicted_neutral)

    # Macro averages
    prec_macro = (neg_precision + pos_precision + neutral_precision) / 3
    rec_macro = (neg_recall + pos_recall + neutral_recall) / 3
    f1_macro = (neg_f1 + pos_f1 + neutral_f1) / 3

    # Micro averages
    # Calculate micro-average precision
    total_true_positive = true_positive_count_neg + true_positive_count_pos + true_positive_count_neutral
    total_false_positive = false_positive_count_neg + false_positive_count_pos + false_positive_count_neutral
    micro_precision = total_true_positive / (total_true_positive + total_false_positive)

    # Calculate micro-average recall
    total_false_negative = false_negative_count_neg + false_negative_count_pos + false_negative_count_neutral
    micro_recall = total_true_positive / (total_true_positive + total_false_negative)

    # Calculate micro-average F1-score
    micro_f1 = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall)

    print(f"Negative precision: {neg_precision:.3f}  Neutral Precision: {neutral_precision:.3f}  Positive precision: {pos_precision:.3f}")
    print(f"Negative recall: {neg_recall:.3f}     Neutral Recall: {neutral_recall:.3f}     Positive recall: {pos_recall:.3f}")
    print(f"Negative F1: {neg_f1:.3f}         Neutral F1: {neutral_f1:.3f}         Positive F1: {pos_f1:.3f}")

    print()
    print("---------------Macro Averages---------------")
    print(f"Precision: {prec_macro:.3f}; Recall: {rec_macro:.3f}; F1: {f1_macro:.3f}")
    print()
    print("---------------Micro Averages---------------")
    print(f"Precision: {micro_precision:.3f}; Recall: {micro_recall:.3f}; F1: {micro_f1:.3f}")

### Vader analysis tweets as is

In [9]:
# Tweets as is
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text)
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2]

In [10]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.515     Neutral Recall: 0.506     Positive recall: 0.884
Negative F1: 0.680         Neutral F1: 0.672         Positive F1: 0.938

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.635; F1: 0.763

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.628; F1: 0.771

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.80      0.51      0.63      1750
           1       0.60      0.51      0.55      1515
           2       0.56      0.88      0.68      1490

    accuracy                           0.63      4755
   macro avg       0.65      0.64      0.62      4755
weighted avg       0.66      0.63      0.62      4755



### Vader analysis tweets lemmatized

In [11]:
# Tweets lemmatized
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, lemmatize=True)
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2]

In [12]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.522     Neutral Recall: 0.488     Positive recall: 0.881
Negative F1: 0.686         Neutral F1: 0.656         Positive F1: 0.936

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.630; F1: 0.760

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.624; F1: 0.768

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.79      0.52      0.63      1750
           1       0.60      0.49      0.54      1515
           2       0.56      0.88      0.68      1490

    accuracy                           0.62      4755
   macro avg       0.65      0.63      0.62      4755
weighted avg       0.65      0.62      0.62      4755



### Vader analysis tweets only adjectives

In [13]:
# Tweets using only adjectives
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, parts_of_speech_to_consider={'ADJ'})
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2]

In [14]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.210     Neutral Recall: 0.892     Positive recall: 0.438
Negative F1: 0.347         Neutral F1: 0.943         Positive F1: 0.609

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.513; F1: 0.633

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.499; F1: 0.665

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.87      0.21      0.34      1750
           1       0.40      0.89      0.56      1515
           2       0.66      0.44      0.53      1490

    accuracy                           0.50      4755
   macro avg       0.65      0.51      0.47      4755
weighted avg       0.66      0.50      0.47      4755



### Vader analysis tweets lemmatized & only adjectives

In [15]:
# Tweets lemmatized and using only adjectives
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, lemmatize=True, parts_of_speech_to_consider={'ADJ'})
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2]

In [16]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.210     Neutral Recall: 0.892     Positive recall: 0.438
Negative F1: 0.347         Neutral F1: 0.943         Positive F1: 0.609

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.513; F1: 0.633

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.499; F1: 0.665

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.87      0.21      0.34      1750
           1       0.40      0.89      0.56      1515
           2       0.66      0.44      0.53      1490

    accuracy                           0.50      4755
   macro avg       0.65      0.51      0.47      4755
weighted avg       0.66      0.50      0.47      4755



### Vader analysis tweets only nouns

In [17]:
# Tweets using only nouns
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, parts_of_speech_to_consider={'NOUN'})
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2]

In [18]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.143     Neutral Recall: 0.817     Positive recall: 0.340
Negative F1: 0.251         Neutral F1: 0.899         Positive F1: 0.507

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.433; F1: 0.552

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.420; F1: 0.591

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.73      0.14      0.24      1750
           1       0.36      0.82      0.50      1515
           2       0.53      0.34      0.41      1490

    accuracy                           0.42      4755
   macro avg       0.54      0.43      0.38      4755
weighted avg       0.55      0.42      0.38      4755



### Vader analysis tweets lemmatized & only nouns

In [19]:
# Tweets lemmatized and using only nouns
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, lemmatize=True, parts_of_speech_to_consider={'NOUN'})
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2] 

In [20]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.157     Neutral Recall: 0.809     Positive recall: 0.331
Negative F1: 0.271         Neutral F1: 0.895         Positive F1: 0.497

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.432; F1: 0.554

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.419; F1: 0.591

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.72      0.16      0.26      1750
           1       0.36      0.81      0.50      1515
           2       0.52      0.33      0.40      1490

    accuracy                           0.42      4755
   macro avg       0.53      0.43      0.39      4755
weighted avg       0.54      0.42      0.38      4755



### Vader analysis tweets only verbs

In [21]:
# Tweets using only verbs
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, parts_of_speech_to_consider={'VERB'})
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2]

In [22]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.288     Neutral Recall: 0.810     Positive recall: 0.343
Negative F1: 0.447         Neutral F1: 0.895         Positive F1: 0.511

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.480; F1: 0.618

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.472; F1: 0.641

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.77      0.29      0.42      1750
           1       0.38      0.81      0.52      1515
           2       0.57      0.34      0.43      1490

    accuracy                           0.47      4755
   macro avg       0.58      0.48      0.46      4755
weighted avg       0.59      0.47      0.45      4755



### Vader analysis tweets lemmatized & only verbs

In [23]:
# Tweets lemmatized and using only verbs
tweets_annotations = []
for i, textb in enumerate(airline_tweets_train.data):
    text = textb.decode('utf-8')
    vader_output = run_vader(text, lemmatize=True, parts_of_speech_to_consider={'VERB'})
    vader_label = vader_output_to_label(vader_output)
    tweets_annotations.append(vader_label)

neg_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 0]
neutral_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 1]
pos_tweets = [tweets_annotations[i] for i in range(len(tweets_annotations)) if airline_tweets_train.target[i] == 2] 

In [24]:
classification_report_own(neg_tweets, pos_tweets, neutral_tweets)
y_pred = [0 if ann == 'negative' else(1 if ann == 'neutral' else 2) for ann in tweets_annotations]
print('\n' + '++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++' + '\n')
print(classification_report(airline_tweets_train.target, y_pred))

Negative precision: 1.000  Neutral Precision: 1.000  Positive precision: 1.000
Negative recall: 0.295     Neutral Recall: 0.780     Positive recall: 0.352
Negative F1: 0.456         Neutral F1: 0.877         Positive F1: 0.520

---------------Macro Averages---------------
Precision: 1.000; Recall: 0.476; F1: 0.618

---------------Micro Averages---------------
Precision: 1.000; Recall: 0.468; F1: 0.637

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

              precision    recall  f1-score   support

           0       0.74      0.30      0.42      1750
           1       0.38      0.78      0.51      1515
           2       0.57      0.35      0.43      1490

    accuracy                           0.47      4755
   macro avg       0.56      0.48      0.46      4755
weighted avg       0.57      0.47      0.45      4755



### Model Comparison Notes
**"Which category performs best, is this the case for any setting?"**

It is interesting to note that for the negative (category '0') high recall in maintained across all settings, which indicates that the model is capable of correctly identifying most of the negative tweets. This is expected, as negative sentiment often includes specific keywords that are very strongly indicative of the sentiment, and are therefore less likely to occur in neutral or positive contexts.
We observe that while for most settings the negative tweets (category '0') has a pretty good f1-score, with the Bag of Words vectorization as min_df of 2 the f1-score is slightly higher for the positive category (category '2'). This suggests that the inclusion of rarer terms (which are not filtered out at lower min_df) is a good choice to capture significant sentiment value for the positive classification.

**"Does the frequency threshold affect the scores? Why or why not according to you?"**

The frequency threshold (min_df) also has an interesting effect on the classifier's performance. Interestingly, under Bag of Words, we find that as we increase min_df from 2 to 5, we get little improvement in precision and f1-score for the neutral ('1') category. Perhaps discarding the infrequent words helps, in that it removes noise and permits the classifier to focus on a set of more salient words that distinguish neutrality from non-neutrality. As we push the threshold even further to min_df of 10, we fail to see much improvement at all. Indeed, the accuracy and macro-average f1-score take a dip. This could be because the infrequent words could also include some very informative features that are necessary for the classifier to understand the nuances within neutrality and between negativity and positivity.
When comparing how the two methods of vectorization perform, essentially TF-IDF and Bag of Words yield the same accuracies. They start to differ when we look at the precision and recall for the neutral ('1') and positive ('2') categories. We generally see that TF-IDF gives us a higher precision for these categories, probably stemming from its ability to identify the words that appear less frequently as more informative, and thus ones that ought to do more to predict that these are neutral and positive sentiments.

---
In summary, the negative category ('0') has the best performance amongst recall and f1-score among several settings reflecting its distinctive nature for negative sentiment expression. The frequency threshold (min_df) does influence classifier scores with a modest impact in the provided data. A moderate min_df can help to cut down on noise, but an overly high threshold risks losing features that are valuable for determining the nuanced differences between classes, particularly for neutral and positive sentiment.

In [25]:
"""
This section compares several vectorization settings in a compact experiment loop.
"""

from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
import nltk
from nltk.corpus import stopwords


tfidf_transformer = TfidfTransformer()
airline_vec = CountVectorizer(min_df=2, # If a token appears fewer times than this, across all documents, it will be ignored
                             tokenizer=nltk.word_tokenize, # we use the nltk tokenizer
                             stop_words=stopwords.words('english')) # stopwords are removed

airline_counts = airline_vec.fit_transform(airline_tweets_train.data)


airline_tfidf = tfidf_transformer.fit_transform(airline_counts)

x_train, x_test, y_train, y_test = train_test_split(airline_tfidf,
                                                   airline_tweets_train.target,
                                                   test_size=0.2)

base_model = MultinomialNB().fit(x_train, y_train)

y_pred = base_model.predict(x_test)

print(classification_report(y_test, y_pred))

/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.84      0.92      0.88       351
           1       0.84      0.72      0.78       309
           2       0.82      0.86      0.84       291

    accuracy                           0.83       951
   macro avg       0.83      0.83      0.83       951
weighted avg       0.84      0.83      0.83       951



#### Using Bag Of Words representation instead of TF-IDF

In [26]:
x_train, x_test, y_train, y_test = train_test_split(airline_counts,
                                                   airline_tweets_train.target,
                                                   test_size=0.2)

base_model = MultinomialNB().fit(x_train, y_train)

y_pred = base_model.predict(x_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.92      0.88       363
           1       0.87      0.76      0.81       303
           2       0.82      0.86      0.84       285

    accuracy                           0.85       951
   macro avg       0.85      0.84      0.84       951
weighted avg       0.85      0.85      0.85       951



#### Using min_df = 5

In [27]:
airline_vec = CountVectorizer(min_df=5, # If a token appears fewer times than this, across all documents, it will be ignored
                             tokenizer=nltk.word_tokenize, # we use the nltk tokenizer
                             stop_words=stopwords.words('english')) # stopwords are removed

airline_counts = airline_vec.fit_transform(airline_tweets_train.data)


airline_tfidf = tfidf_transformer.fit_transform(airline_counts)

x_train, x_test, y_train, y_test = train_test_split(airline_tfidf,
                                                   airline_tweets_train.target,
                                                   test_size=0.2)

base_model = MultinomialNB().fit(x_train, y_train)

y_pred = base_model.predict(x_test)

print(classification_report(y_test, y_pred))

/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.79      0.92      0.85       337
           1       0.86      0.67      0.76       320
           2       0.81      0.87      0.84       294

    accuracy                           0.82       951
   macro avg       0.82      0.82      0.82       951
weighted avg       0.82      0.82      0.82       951



#### Using min_df = 10

In [28]:
airline_vec = CountVectorizer(min_df=10, # If a token appears fewer times than this, across all documents, it will be ignored
                             tokenizer=nltk.word_tokenize, # we use the nltk tokenizer
                             stop_words=stopwords.words('english')) # stopwords are removed

airline_counts = airline_vec.fit_transform(airline_tweets_train.data)


airline_tfidf = tfidf_transformer.fit_transform(airline_counts)

x_train, x_test, y_train, y_test = train_test_split(airline_tfidf,
                                                   airline_tweets_train.target,
                                                   test_size=0.2)

base_model = MultinomialNB().fit(x_train, y_train)

y_pred = base_model.predict(x_test)

print(classification_report(y_test, y_pred))

/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.82      0.87      0.84       330
           1       0.81      0.72      0.76       322
           2       0.80      0.84      0.82       299

    accuracy                           0.81       951
   macro avg       0.81      0.81      0.81       951
weighted avg       0.81      0.81      0.81       951



In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
import nltk

airline_vec = CountVectorizer(min_df=2, # If a token appears fewer times than this, across all documents, it will be ignored
                             tokenizer=nltk.word_tokenize, # we use the nltk tokenizer
                             stop_words=stopwords.words('english')) # stopwords are removed


# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    airline_tweets_train.data,
    airline_tweets_train.target,
    test_size=0.20
)

vectorizers = {
    'TF-IDF': TfidfVectorizer,
    'Bag of Words': CountVectorizer
}

min_dfs = [2, 5, 10]

# Iterate over different settings
for vectorizer_name, vectorizer_class in vectorizers.items():
    for min_df in min_dfs:
        # Define vectorizer with current settings
        vectorizer = vectorizer_class(min_df=min_df, tokenizer=nltk.word_tokenize, stop_words='english')

        # Vectorize training and test data
        X_train_vectorized = vectorizer.fit_transform(X_train)
        X_test_vectorized = vectorizer.transform(X_test)

        # Train Naive Bayes classifier
        clf = MultinomialNB()
        clf.fit(X_train_vectorized, y_train)

        # Predict on test data
        y_pred = clf.predict(X_test_vectorized)

        # Evaluate performance
        print(f"Vectorizer: {vectorizer_name}, min_df: {min_df}")
        print(classification_report(y_test, y_pred))
        print("-------------------------------------------------------")

/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vectorizer: TF-IDF, min_df: 2
              precision    recall  f1-score   support

           0       0.79      0.93      0.86       354
           1       0.87      0.68      0.76       303
           2       0.83      0.85      0.84       294

    accuracy                           0.83       951
   macro avg       0.83      0.82      0.82       951
weighted avg       0.83      0.83      0.82       951

-------------------------------------------------------


/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vectorizer: TF-IDF, min_df: 5
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       354
           1       0.83      0.73      0.78       303
           2       0.84      0.84      0.84       294

    accuracy                           0.83       951
   macro avg       0.83      0.82      0.82       951
weighted avg       0.83      0.83      0.83       951

-------------------------------------------------------


/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


...

Vectorizer: Bag of Words, min_df: 2
              precision    recall  f1-score   support

           0       0.82      0.92      0.87       354
           1       0.85      0.73      0.79       303
           2       0.86      0.85      0.86       294

    accuracy                           0.84       951
   macro avg       0.84      0.83      0.84       951
weighted avg       0.84      0.84      0.84       951

-------------------------------------------------------


/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vectorizer: Bag of Words, min_df: 5
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       354
           1       0.84      0.74      0.79       303
           2       0.85      0.85      0.85       294

    accuracy                           0.83       951
   macro avg       0.84      0.83      0.83       951
weighted avg       0.83      0.83      0.83       951

-------------------------------------------------------


/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Vectorizer: Bag of Words, min_df: 10
              precision    recall  f1-score   support

           0       0.81      0.91      0.85       354
           1       0.83      0.75      0.79       303
           2       0.85      0.81      0.83       294

    accuracy                           0.83       951
   macro avg       0.83      0.82      0.82       951
weighted avg       0.83      0.83      0.82       951

-------------------------------------------------------


### Feature Interpretation
1. **Expected Features for Each Class**:
   - For the negative class we would expect words associated with negative sentiment, such as 'delayed', 'cancelled', 'worst', 'late', 'lost', or similar. Also, terms related to negative experiences like 'bad', 'never', 'waiting'. So overall we would expect words that we generally associate with negative experiences while travelling by plane, such as when a plane is delayed or cancelled, or the stewards offer a bad service, or the food on the plane is bad/very expensive. 
   - For the neutral class we would expect terms that are likely to appear in tweets regardless of sentiment, such as mentions of airlines ('united', 'jetblue', 'southwestair', 'americanair', 'usairways', 'virginamerica'), general words or questions ('?', 'please', 'help', 'need'), and links ('http'). So we would expect less words related to sentiment, such as 'good', 'bad', or similar words, in this class, because if those words were in the tweet they would probably be labelled with either a positive or negative sentiment.
   - For the positive class we would expect words conveying positive sentiment, appreciation, or satisfaction, such as 'thanks', 'great', 'awesome', 'love', 'much', 'amazing', etc. We could aslo expect words such as 'on', 'time' as if they are in the same tweet they most probably appear contiguously expressing the phrase 'on tiime' which is a good thing when talking about airplane flights. We could even expect words like, 'delicious', 'tasty', etc., which would express a positive sentiment related to the food served during the flight.

2. **Unexpected Features**:
   - For the negative class, the unexpected features might include airline names or punctuation marks that are not directly related to negative sentiment, like '@', '.', '?', '!', '#', etc. Generally this features would be unexpected as they are not directly linked to a sentiment, like the special characters, such as '?', '!', or '@'.
   - For the neutral class, the unexpected features could be specific terms or airline names that are typically associated with either positive or negative sentiment, but in this context, they appear in neutral tweets. For instance, 'southwestair', 'jetblue', 'united', 'americanair', etc. Generally, people that write tweets mentioning a specific airline it is to express a either good or bad experience with that airline, so it is unexpected to find this kinds of features in the 80 most important features for the neutral class.
   - For the positive class, the unexpected features could be terms that may appear in positive tweets but are not directly indicative of positive sentiment, such as 'us', 'response', 'home', 'back', 'work', etc. Another unexpected feature for the positive class is "n't". We find this feature unexpected as it is usually used in words like "can't", or "don't" which can be widely used in negatively labeled tweets.

3. **Words to Remove or Keep**:
   - Words we can remove are words that are not informative for sentiment analysis, such as airline names ('united', 'jetblue', etc.), punctuation marks ('@', '.', '?', '!', etc.), and generic terms ('http', 'please', 'need', 'thanks', etc.). For example, the feature 'virginamerica' appears in the important feature lists of all tweet classes. This shows that the names of the airlines are not very informative for sentiment analysis and if they are related to the positive/negative classes it will be very dependent on the collection of tweets gathered. With a big enough collection of tweets all airlines will be associated with all 3 tweets classes, so they could be removed for sentiment analysis. Similarly you could apply this same logic to the punctuation marks and generic words exampled previously. By removing this words to carry out the sentiment analysis of the tweets we can get rid of useless features. This could possibly allow for more uncommon features to be included in the features for sentiment analysis allowing the model to capture a more accurate depiction of positive and negative tweets increasing performance.
   - Words we can keep are words that are indicative of sentiment or provide context, such as 'delayed', 'cancelled', 'worst', 'late', 'lost' for negative sentiment, and 'thanks', 'great', 'awesome', 'love', 'much', 'amazing' for positive sentiment. By keeping this words the model can better capture the nuances of the patterns and words used in positive and negative tweets, which would in turn increase the performance of the model.

In [30]:
from sklearn.feature_extraction.text import CountVectorizer

def important_features_per_class(vectorizer,classifier,n=80):
    class_labels = classifier.classes_
    feature_names = vectorizer.get_feature_names_out()
    topn_class1 = sorted(zip(classifier.feature_count_[0], feature_names),reverse=True)[:n]
    topn_class2 = sorted(zip(classifier.feature_count_[1], feature_names),reverse=True)[:n]
    topn_class3 = sorted(zip(classifier.feature_count_[2], feature_names),reverse=True)[:n]
    print("Important words in negative documents")
    for coef, feat in topn_class1:
        print(class_labels[0], coef, feat)
    print("-----------------------------------------")
    print("Important words in neutral documents")
    for coef, feat in topn_class2:
        print(class_labels[1], coef, feat)
    print("-----------------------------------------")
    print("Important words in positive documents")
    for coef, feat in topn_class3:
        print(class_labels[2], coef, feat)

# Example call:
#important_features_per_class(airline_vec, clf)

# Define CountVectorizer with min_df=2
airline_vec = CountVectorizer(min_df=2, # If a token appears fewer times than this, across all documents, it will be ignored
                             tokenizer=nltk.word_tokenize, # we use the nltk tokenizer
                             stop_words=stopwords.words('english')) # stopwords are removed

airline_counts = airline_vec.fit_transform(airline_tweets_train.data)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    airline_counts,
    airline_tweets_train.target,
    test_size=0.20
)
# Train Naive Bayes classifier
clf = MultinomialNB()
clf.fit(X_train, y_train)

important_features_per_class(airline_vec, clf)

/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/javi/anaconda3/envs/TM/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'d", "'ll", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'] not in stop_words.
  warnings.warn(


Important words in negative documents
0 1519.0 @
0 1390.0 united
0 1230.0 .
0 443.0 ``
0 405.0 ?
0 404.0 flight
0 361.0 !
0 321.0 #
0 222.0 n't
0 154.0 ''
0 125.0 's
0 113.0 :
0 111.0 virginamerica
0 108.0 service
0 97.0 get
0 95.0 delayed
0 92.0 customer
0 91.0 bag
0 87.0 cancelled
0 85.0 plane
0 82.0 time
0 77.0 ;
0 76.0 -
0 71.0 http
0 71.0 gate
0 71.0 ...
0 70.0 &
0 69.0 still
0 69.0 hours
0 69.0 'm
0 68.0 late
0 65.0 hour
0 60.0 2
0 59.0 amp
0 57.0 help
0 56.0 would
0 55.0 airline
0 52.0 flights
0 52.0 ca
0 48.0 waiting
0 48.0 one
0 46.0 delay
0 44.0 never
0 44.0 like
0 44.0 3
0 43.0 worst
0 43.0 (
0 42.0 lost
0 42.0 flightled
0 42.0 back
0 42.0 $
0 41.0 really
0 41.0 due
0 41.0 )
0 41.0 've
0 40.0 seat
0 40.0 luggage
0 39.0 wait
0 39.0 us
0 38.0 fly
0 37.0 u
0 37.0 check
0 37.0 another
0 36.0 thanks
0 36.0 bags
0 35.0 day
0 34.0 people
0 34.0 ever
0 34.0 crew
0 34.0 baggage
0 33.0 hold
0 33.0 airport
0 32.0 seats
0 32.0 last
0 32.0 4
0 30.0 trying
0 30.0 today
0 30.0 days
0 29.0 